# Descriptive Statistics of the Corpus

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = 'data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

In [4]:
fs = grab_all_files(DATA_PATH, '.tsv')
fs

['data/lme_ready/cha-ceda-analysis.tsv',
 'data/lme_ready/candor-ceda-analysis.tsv',
 'data/lme_ready/cha-CABNC-ceda-analysis.tsv']

In [8]:
stats = []

In [9]:
for f in tqdm(fs):
    print(f)
    df = pd.read_table(f,sep='\t')
    
    if '/cha-' in f:
        df['conversation_id'] = ['-'.join(sp.split('-')[:-1]) for sp in tqdm(df['x_speaker'])]
   
    elif '/candor-' in f:
        df['conversation_id'] = [sp.replace('.csv', '') for sp in tqdm(df['conversation_id'])]
        df['corpus'] = 'CANDOR'
    
    else:
        df['conversation_id'] = f
    
    df['x_turn_id_'] = df['conversation_id'] + '-' + df['x_turn_id'].astype(str)
    df['y_turn_id_'] = df['conversation_id'] + '-' + df['y_turn_id'].astype(str)
    
    stats += [
        {
            'file': f,
            'size': 'all',
            'comparisons': len(df),
            'speakers': len(np.unique(df[['x_speaker', 'y_speaker']].values)),
            'conversations': df['conversation_id'].nunique(),
            'utterances': len(np.unique(df[['x_turn_id_', 'y_turn_id_']].values)),
            'k_corpora': df['corpus'].nunique(),
            'corpora': str(df['corpus'].unique().tolist())
        }
    ]
    
    # process
    df = df.loc[df['x_turn_id'] >= 5]
    
    stats += [
        {
            'file': f,
            'size': 'after fifth turn',
            'comparisons': len(df),
            'speakers': len(np.unique(df[['x_speaker', 'y_speaker']].values)),
            'conversations': df['conversation_id'].nunique(),
            'utterances': len(np.unique(df[['x_turn_id', 'y_turn_id']].values)),
            'k_corpora': df['corpus'].nunique(),
            'corpora': str(df['corpus'].unique().tolist())
        }
    ]

    # process
    df = df.loc[
        (df['nx'] >= 5) 
        & (df['ny'] >= 5) 
    ]
    
    stats += [
        {
            'file': f,
            'size': 'fully processed',
            'comparisons': len(df),
            'speakers': len(np.unique(df[['x_speaker', 'y_speaker']].values)),
            'conversations': df['conversation_id'].nunique(),
            'utterances': len(np.unique(df[['x_turn_id', 'y_turn_id']].values)),
            'k_corpora': df['corpus'].nunique(),
            'corpora': str(df['corpus'].unique().tolist())
        }
    ]
    
    del df

  0%|          | 0/3 [00:00<?, ?it/s]

data/lme_ready/cha-ceda-analysis.tsv


  0%|          | 0/8028150 [00:00<?, ?it/s]

data/lme_ready/candor-ceda-analysis.tsv


  0%|          | 0/429480 [00:00<?, ?it/s]

data/lme_ready/cha-CABNC-ceda-analysis.tsv


  0%|          | 0/12723777 [00:00<?, ?it/s]

In [11]:
stats = pd.DataFrame(stats)
stats.head(n=100)

,file,size,comparisons,speakers,conversations,utterances,k_corpora,corpora
0,data/lme_ready/cha-ceda-analysis.tsv,all,8028150,1068,375,145104,10,"['GCSAusE', 'SBCSAE', 'SCoSE', 'callfriend-eng..."
1,data/lme_ready/cha-ceda-analysis.tsv,after fifth turn,7940100,1060,374,3551,10,"['GCSAusE', 'SBCSAE', 'SCoSE', 'callfriend-eng..."
2,data/lme_ready/cha-ceda-analysis.tsv,fully processed,4119659,1027,374,3006,10,"['GCSAusE', 'SBCSAE', 'SCoSE', 'callfriend-eng..."
3,data/lme_ready/candor-ceda-analysis.tsv,all,429480,68,34,11434,1,['CANDOR']
4,data/lme_ready/candor-ceda-analysis.tsv,after fifth turn,422680,68,34,672,1,['CANDOR']
5,data/lme_ready/candor-ceda-analysis.tsv,fully processed,281817,68,34,637,1,['CANDOR']
6,data/lme_ready/cha-CABNC-ceda-analysis.tsv,all,12723777,6168,1998,258636,60,"['CABNC-0missing', 'CABNC-KB0', 'CABNC-KB1', '..."
7,data/lme_ready/cha-CABNC-ceda-analysis.tsv,after fifth turn,12435395,5721,1865,1598,60,"['CABNC-0missing', 'CABNC-KB0', 'CABNC-KB1', '..."
8,data/lme_ready/cha-CABNC-ceda-analysis.tsv,fully processed,5857285,5161,1819,1562,60,"['CABNC-0missing', 'CABNC-KB0', 'CABNC-KB1', '..."


In [12]:
stats.to_csv(os.path.join(DATA_PATH,'corpus-descriptives.csv'), index=False, encoding='utf-8')